# Task 5 - First Attempt: google/gemma-2-9b-it

Task 2 used `meta-llama/Meta-Llama-3.1-8B-Instruct`. Following the assignment instructions, this notebook tries the other model from the Task 2 list - `google/gemma-2-9b-it` - as the judge.

The sanity check below runs the judge on 5 products. Based on the results, I document whether this model is suitable and whether a switch is needed.

## Imports & Configuration

In [1]:
import os
import time
import pandas as pd
from typing import Literal
from pydantic import BaseModel
from openai import OpenAI

API_KEY     = os.getenv("NEBIUS_API_KEY")
BASE_URL    = "https://api.tokenfactory.nebius.com/v1/"
# google/gemma-2-9b-it is not available on the Nebius Token Factory API.
# The closest available variant is google/gemma-2-9b-it-fast.
JUDGE_MODEL = "google/gemma-2-9b-it-fast"

XLSX_PATH      = os.path.join(os.path.dirname(os.path.abspath("__file__")), "assignment_01.xlsx")
JUDGE_CRITERIA = ["fluency", "grammar", "tone", "length", "grounding"]

## Pydantic Schema & Judge Prompt

In [2]:
class CriterionRating(BaseModel):
    explanation: str          # reasoning comes first — forces the model to analyse before concluding
    verdict: Literal["good", "ok", "bad"]


class JudgeOutput(BaseModel):
    fluency:   CriterionRating
    grammar:   CriterionRating
    tone:      CriterionRating
    length:    CriterionRating
    grounding: CriterionRating


JUDGE_SYSTEM_PROMPT = """\
You are an expert evaluator of e-commerce product descriptions. \
Your task is to rate a generated product description against five quality criteria.

For each criterion, you must provide:
  1. explanation — your reasoning (analyse the text, cite specific phrases if relevant)
  2. verdict     — one of: good | ok | bad

Always write the explanation before the verdict. Do not pick a verdict first and then justify it.

=== RUBRIC ===

FLUENCY
  good : The description reads naturally and engagingly. Sentences flow well, transitions are smooth, \
and the writing feels polished.
  ok   : Readable overall, but contains minor awkward phrases, repetition, or slightly choppy transitions \
that interrupt the flow.
  bad  : Difficult to read. Contains confusing structure, very unnatural phrasing, or reads like a \
rough draft.

GRAMMAR
  good : No spelling, punctuation, or grammatical errors.
  ok   : One or two minor errors (typo, missing comma, minor agreement issue) that do not impede \
understanding.
  bad  : Multiple errors, or errors that make the text confusing or unprofessional.

TONE
  good : Warm, confident, and benefit-focused. Leads with what the customer gains. Avoids dry spec \
lists and hollow hype words ("amazing", "revolutionary").
  ok   : Adequate but imperfect — too neutral/dry, or slightly over-hyped, or mixes benefit-focus \
with spec-list language.
  bad  : Inappropriate register — cold, arrogant, sarcastic, or wildly mismatched to a retail context.

LENGTH
  good : Between 50 and 90 words (inclusive).
  ok   : Between 40–49 words or 91–110 words.
  bad  : 39 words or fewer, or 111 words or more.
Count words carefully before assigning a verdict.

GROUNDING
  good : Every factual claim in the description is directly supported by the product information provided.
  ok   : The description makes a minor reasonable inference not explicitly stated but plausible given \
the product data.
  bad  : The description invents at least one feature, material, specification, or fact that is NOT \
present in the product information.
IMPORTANT: Marketing language ("lightning-fast", "stunning", "powerhouse", "breathtaking") applied \
to real, listed features is NOT a grounding failure. Only penalise fabricated facts.
"""


def build_judge_message(row: pd.Series) -> str:
    return (
        "=== PRODUCT INFORMATION ===\n"
        f"Product name : {row['product_name']}\n"
        f"Attributes   : {row['Product_attribute_list']}\n"
        f"Material     : {row['material']}\n"
        f"Warranty     : {row['warranty']}\n"
        "\n"
        "=== GENERATED DESCRIPTION ===\n"
        f"{row['generated_description']}\n"
        "\n"
        "Evaluate the description against all five criteria (fluency, grammar, tone, length, grounding). "
        "For each criterion write your explanation first, then your verdict."
    )


def judge_description(client: OpenAI, row: pd.Series) -> dict:
    """
    Call the judge model for one product row.
    Returns a flat dict with keys like fluency_explanation, fluency_verdict, etc.
    """
    try:
        response = client.beta.chat.completions.parse(
            model=JUDGE_MODEL,
            messages=[
                {"role": "system", "content": JUDGE_SYSTEM_PROMPT},
                {"role": "user",   "content": build_judge_message(row)},
            ],
            response_format=JudgeOutput,
            temperature=0.1,   # low temperature for consistent, reproducible judgements
            max_tokens=2048,
        )
        result: JudgeOutput = response.choices[0].message.parsed
        flat = {}
        for criterion in JUDGE_CRITERIA:
            rating = getattr(result, criterion)
            flat[f"{criterion}_explanation"] = rating.explanation
            flat[f"{criterion}_verdict"]     = rating.verdict
        flat["judge_error"] = ""
        return flat
    except Exception as e:
        flat = {}
        for criterion in JUDGE_CRITERIA:
            flat[f"{criterion}_explanation"] = ""
            flat[f"{criterion}_verdict"]     = ""
        flat["judge_error"] = str(e)
        return flat

## Sanity Check - 5 Products

In [3]:
SANITY_ROWS = [0, 3, 9, 13, 14]   # iPhone, Sony headphones, Garmin, Nintendo Switch, PS5

df_baseline = pd.read_excel(XLSX_PATH, sheet_name="baseline")
client      = OpenAI(api_key=API_KEY, base_url=BASE_URL)

print("=" * 65)
print("TASK 5 — First Attempt Sanity Check (google/gemma-2-9b-it-fast)")
print("=" * 65)
print(f"Judge model: {JUDGE_MODEL}\n")

for idx in SANITY_ROWS:
    row = df_baseline.iloc[idx]
    print(f"[{idx:02d}] {row['product_name']}")
    start  = time.time()
    result = judge_description(client, row)
    elapsed = round((time.time() - start) * 1000)

    if result["judge_error"]:
        print(f"  ERROR: {result['judge_error']}\n")
        continue

    for criterion in JUDGE_CRITERIA:
        v   = result.get(f"{criterion}_verdict", "")
        exp = result.get(f"{criterion}_explanation", "")
        print(f"  {criterion:<10} [{v}]  {str(exp)[:110]}")
    print(f"  latency: {elapsed} ms\n")

TASK 5 — First Attempt Sanity Check (google/gemma-2-9b-it-fast)
Judge model: google/gemma-2-9b-it-fast

[00] Apple iPhone 15 Pro
  fluency    [good]  The description reads smoothly and engagingly. Sentences flow well, and transitions like 'powered by' and 'thi
  grammar    [good]  There are no grammatical errors. Punctuation is correct, and sentence structure is sound.
  tone       [good]  The tone is enthusiastic and benefit-focused. It highlights the phone's performance, durability, and compact d
  length     [good]  The description is 90 words long.
  grounding  [good]  All claims are supported by the product information. 'Lightning-fast A17 Pro chip,' '120Hz ProMotion display,'
  latency: 7351 ms

[03] Sony WH‑1000XM5 Headphones
  fluency    [good]  The description reads smoothly and naturally. Sentences flow well together, and transitions are clear. For exa
  grammar    [good]  There are no spelling, punctuation, or grammatical errors in the description.
  tone       [good]  The t

## Conclusion

**`google/gemma-2-9b-it-fast` is not reliable enough for automated evaluation.**

> Note: `google/gemma-2-9b-it` is not available on the Nebius Token Factory API.  
> The closest available variant, `google/gemma-2-9b-it-fast`, was used instead.  
> The Pydantic schema (`JudgeOutput`) and judge prompt are **identical** to those used in the
> final implementation (`task5_judge.ipynb`) — verified by string comparison.

### Attempt 1 - `max_tokens=1024`

4 out of 5 calls completed, but with issues:

| Product | Outcome | Issue |
|---|---|---|
| Apple iPhone 15 Pro | All verdicts returned | `length=ok` for 91 words (borderline - ok range is 91–110) |
| Sony WH-1000XM5 | All verdicts returned | — |
| Garmin Forerunner 255 | **ERROR** — token limit reached | Consumed all 1 024 output tokens |
| Nintendo Switch OLED | All verdicts returned | **Wrong verdict**: `length=bad` for 84 words (rubric: `good` at 50–90) |
| PlayStation 5 Slim | All verdicts returned | Latency: 8 728 ms |

### Attempt 2 - `max_tokens=2048`

Doubling the token budget did not resolve the core problems:

| Product | Outcome | Issue |
|---|---|---|
| Apple iPhone 15 Pro | All verdicts returned | Latency: 7 351 ms |
| Sony WH-1000XM5 | All verdicts returned | **Wrong verdict**: `length=ok` for 82 words (rubric: `good` at 50–90) |
| Garmin Forerunner 255 | **ERROR** — token limit reached | Still fails even at 2 048 tokens |
| Nintendo Switch OLED | All verdicts returned | **Wrong verdict**: `length=ok` for 80 words (rubric: `good` at 50–90) |
| PlayStation 5 Slim | All verdicts returned | — |

Three persistent problems across both attempts:

1. **Reliability** - Garmin fails at both `max_tokens=1024` and `max_tokens=2048`. For longer
   prompts the model generates verbose prose before writing JSON, exhausting the token budget
   before completing a parseable structure.

2. **Verdict accuracy** — The length criterion is wrong in 3 out of 5 products across the two
   runs. Nintendo Switch got `bad` for 84 words, then `ok` for 80 words on the retry — both
   are factually incorrect (50–90 words is `good`). The model miscounts words inconsistently.

3. **Latency** — Responses ranged from 2 542 ms to 8 728 ms across the same 5 products,
   compared to a stable 4–6 s for Qwen.

**Decision:** Switch to `Qwen/Qwen3-30B-A3B-Instruct-2507`.

Qwen3-30B returned correct, parseable structured output on all 5 sanity-check products with
no verdict errors and consistent latency. It is from a different architecture family than the
Llama generator used in Task 2, reducing shared-bias risk. See `task5_judge.ipynb` for the
full implementation and results.